# Feature Engineering : House Prices

## Objectif
Transformer les variables brutes en features exploitables par le modèle de régression linéaire : encodage des variables catégorielles, création de variables dérivées, préparation finale du dataset d'entraînement.

## Étapes
1. Chargement du dataset nettoyé
2. Encodage ordinal des variables de qualité
3. Encodage one-hot des variables catégorielles nominales
4. Vérification de la multicolinéarité
5. Sélection finale des variables
6. Export du dataset prêt pour la modélisation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/train_cleaned.csv', keep_default_na=False, na_values=[''])
print(df.shape)
df.head()

(1458, 82)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,SalePrice_log
0,1,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,2,2008,WD,Normal,208500,12.247699
1,2,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,5,2007,WD,Normal,181500,12.109016
2,3,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,9,2008,WD,Normal,223500,12.317171
3,4,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,2,2006,WD,Abnorml,140000,11.849405
4,5,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,12,2008,WD,Normal,250000,12.429220


In [2]:
quality_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
for col in quality_cols:
    df[col] = df[col].map(quality_mapping)

In [3]:
df[quality_cols].head()

,ExterQual,ExterCond,BsmtQual,BsmtCond,HeatingQC,KitchenQual,FireplaceQu,GarageQual,GarageCond,PoolQC
0,4,3,4,3,5,4,0,3,3,0
1,3,3,4,3,5,3,3,3,3,0
2,4,3,4,3,5,4,3,3,3,0
3,3,3,3,4,4,4,4,3,3,0
4,4,3,4,3,5,4,3,3,3,0


In [4]:
remaining_categorical = df.select_dtypes(include='object').columns
print(remaining_categorical.tolist())
print("\n",len(remaining_categorical), "variables categorielles restantes")

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']

 33 variables categorielles restantes


In [5]:
for col in remaining_categorical:
    print(col, ':', sorted(df[col].unique()))

MSZoning : ['C (all)', 'FV', 'RH', 'RL', 'RM']
Street : ['Grvl', 'Pave']
Alley : ['Grvl', 'None', 'Pave']
LotShape : ['IR1', 'IR2', 'IR3', 'Reg']
LandContour : ['Bnk', 'HLS', 'Low', 'Lvl']
Utilities : ['AllPub', 'NoSeWa']
LotConfig : ['Corner', 'CulDSac', 'FR2', 'FR3', 'Inside']
LandSlope : ['Gtl', 'Mod', 'Sev']
Neighborhood : ['Blmngtn', 'Blueste', 'BrDale', 'BrkSide', 'ClearCr', 'CollgCr', 'Crawfor', 'Edwards', 'Gilbert', 'IDOTRR', 'MeadowV', 'Mitchel', 'NAmes', 'NPkVill', 'NWAmes', 'NoRidge', 'NridgHt', 'OldTown', 'SWISU', 'Sawyer', 'SawyerW', 'Somerst', 'StoneBr', 'Timber', 'Veenker']
Condition1 : ['Artery', 'Feedr', 'Norm', 'PosA', 'PosN', 'RRAe', 'RRAn', 'RRNe', 'RRNn']
Condition2 : ['Artery', 'Feedr', 'Norm', 'PosA', 'PosN', 'RRAe', 'RRAn', 'RRNn']
BldgType : ['1Fam', '2fmCon', 'Duplex', 'Twnhs', 'TwnhsE']
HouseStyle : ['1.5Fin', '1.5Unf', '1Story', '2.5Fin', '2.5Unf', '2Story', 'SFoyer', 'SLvl']
RoofStyle : ['Flat', 'Gable', 'Gambrel', 'Hip', 'Mansard', 'Shed']
RoofMatl : ['Com

In [7]:
quality_scale = {'None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'}

for col in remaining_categorical:
    values = set(df[col].unique())
    if values <= quality_scale:
        print(col, ': même échelle qualité -> ordinal facile')
    elif df[col].nunique() == 2:
        print('\n',col, ': binaire -> encodage 0/1 direct')
    else:
        print('\n',col, ': candidate au one-hot (pas de pattern connu)')


 MSZoning : candidate au one-hot (pas de pattern connu)

 Street : binaire -> encodage 0/1 direct

 Alley : candidate au one-hot (pas de pattern connu)

 LotShape : candidate au one-hot (pas de pattern connu)

 LandContour : candidate au one-hot (pas de pattern connu)

 Utilities : binaire -> encodage 0/1 direct

 LotConfig : candidate au one-hot (pas de pattern connu)

 LandSlope : candidate au one-hot (pas de pattern connu)

 Neighborhood : candidate au one-hot (pas de pattern connu)

 Condition1 : candidate au one-hot (pas de pattern connu)

 Condition2 : candidate au one-hot (pas de pattern connu)

 BldgType : candidate au one-hot (pas de pattern connu)

 HouseStyle : candidate au one-hot (pas de pattern connu)

 RoofStyle : candidate au one-hot (pas de pattern connu)

 RoofMatl : candidate au one-hot (pas de pattern connu)

 Exterior1st : candidate au one-hot (pas de pattern connu)

 Exterior2nd : candidate au one-hot (pas de pattern connu)

 MasVnrType : candidate au one-hot (pa

In [8]:
df_encoded = pd.get_dummies(df, columns=remaining_categorical, drop_first=True)
print(df_encoded.shape)

(1458, 231)


In [9]:
corr_pairs = df_encoded[['GarageCars', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 
                          'GrLivArea', 'TotRmsAbvGrd', 'YearBuilt', 'GarageYrBlt']].corr()
corr_pairs

,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,GrLivArea,TotRmsAbvGrd,YearBuilt,GarageYrBlt
GarageCars,1.000000,0.887304,0.451890,0.449195,0.475442,0.361152,0.537301,0.598211
GarageArea,0.887304,1.000000,0.475069,0.477299,0.456358,0.328714,0.477998,0.563999
TotalBsmtSF,0.451890,0.475069,1.000000,0.803830,0.408793,0.266146,0.400266,0.182964
1stFlrSF,0.449195,0.477299,0.803830,1.000000,0.533697,0.396381,0.281253,0.170177
GrLivArea,0.475442,0.456358,0.408793,0.533697,1.000000,0.829498,0.194397,0.165287
TotRmsAbvGrd,0.361152,0.328714,0.266146,0.396381,0.829498,1.000000,0.091220,0.094946
YearBuilt,0.537301,0.477998,0.400266,0.281253,0.194397,0.091220,1.000000,0.271812
GarageYrBlt,0.598211,0.563999,0.182964,0.170177,0.165287,0.094946,0.271812,1.000000


In [10]:
cols_to_drop = ['GarageArea', '1stFlrSF', 'TotRmsAbvGrd']
df_encoded = df_encoded.drop(columns=cols_to_drop)
print(df_encoded.shape)

(1458, 228)


In [12]:
df_final = df_encoded.drop(columns=['Id', 'SalePrice'])
print(df_final.shape)
df_final.columns.tolist()[:10]

(1458, 226)


['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'ExterQual',
 'ExterCond']

In [13]:
df_final['MSSubClass'].dtype

dtype('int64')

In [14]:
df_final['MSSubClass'] = df_final['MSSubClass'].astype(str)
df_final = pd.get_dummies(df_final, columns=['MSSubClass'], drop_first=True)
print(df_final.shape)

(1458, 239)


In [15]:
df_final.to_csv('../data/processed/train_final.csv', index=False)
print("Fichier exporte avec succes :", df_final.shape)

Fichier exporte avec succes : (1458, 239)
